# 03 — Embedding Experiments

## Embedding Models + Multi-Question Retrieval Evaluation

**Experiment IDs:** EMB-001, EMB-002

### Objective

Compare OpenAI embedding models using the selected chunking baseline from Notebook 02 and evaluate retrieval behavior across multiple non-trivial business questions.

### Baseline

```text
Dataset       : Pharma Sales CSV
Documents     : 300
Chunk Size    : 500
Chunk Overlap : 50
Chunks        : 1,538
```

### Models

| Experiment | Model | Purpose |
|---|---|---|
| EMB-001 | `text-embedding-3-small` | Cost-efficient baseline |
| EMB-002 | `text-embedding-3-large` | Higher-capability comparison |

### Evaluation

Each model is tested against four deliberately non-trivial business questions.

The same documents, chunks, questions and `k` are used for both models.

Only the embedding model changes.

## 1. Environment & Imports

This notebook uses the OpenAI embedding API through LangChain.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY environment variable is not set. "
        "Please configure it in your .env file."
    )

print("Environment configured successfully.")

C:\Users\visha\AppData\Local\Temp\ipykernel_38684\2646475630.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


Environment configured successfully.


## 2. Load the Canonical Dataset

The same dataset used in Notebooks 01 and 02 is loaded explicitly so this notebook remains independently reproducible.

In [2]:
DATA_PATH = "../data/Pharma_Sales_Long.csv"

loader = CSVLoader(
    file_path=DATA_PATH,
    encoding="utf-8"
)

data = loader.load()

assert len(data) == 300, (
    f"Expected 300 documents, but found {len(data)}."
)

print(f"Loaded {len(data)} documents.")

Loaded 300 documents.


## 3. Recreate the Chunking Baseline

Notebook 02 selected `CHK-002`:

```text
Chunk size    : 500
Chunk overlap : 50
Expected      : 1,538 chunks
```

Both embedding models must receive exactly the same chunks for a fair comparison.

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(data)

assert len(chunks) == 1538, (
    f"Expected 1,538 chunks from CHK-002, "
    f"but found {len(chunks)}."
)

print(f"Documents : {len(data)}")
print(f"Chunks    : {len(chunks)}")
print("✓ Chunking baseline validated.")

Documents : 300
Chunks    : 1538
✓ Chunking baseline validated.


## 4. Define Tricky Evaluation Questions

A simple question such as *"What is WELIREG?"* is not enough for a meaningful embedding comparison.

The questions below combine multiple business concepts.

| ID | Question Type | What It Tests |
|---|---|---|
| Q001 | Product + indication + engagement | Multi-attribute semantic matching |
| Q002 | Vaccine + clinical + sales activity | Cross-topic semantic matching |
| Q003 | Territory + opportunity + behavior | Business-context retrieval |
| Q004 | Market access + competitor + compliance | Detailed Notes retrieval |

### Q001
Which sales records describe WELIREG discussions related to renal cell carcinoma in territories where customer engagement or follow-up activity was also mentioned?

### Q002
Find records where the sales discussion combines physician engagement, approved clinical information, and follow-up planning for a vaccine product.

### Q003
Which records indicate regional or territory-level business opportunities while also referring to prescription trends and customer behavior?

### Q004
Find records where customer discussions include market access or competitor comparison together with compliant promotional or scientific-literature activities.

In [4]:
evaluation_questions = {
    "Q001": (
        "Which sales records describe WELIREG discussions related to renal cell "
        "carcinoma in territories where customer engagement or follow-up activity "
        "was also mentioned?"
    ),
    "Q002": (
        "Find records where the sales discussion combines physician engagement, "
        "approved clinical information, and follow-up planning for a vaccine product."
    ),
    "Q003": (
        "Which records indicate regional or territory-level business opportunities "
        "while also referring to prescription trends and customer behavior?"
    ),
    "Q004": (
        "Find records where customer discussions include market access or competitor "
        "comparison together with compliant promotional or scientific-literature activities."
    ),
}

for question_id, question in evaluation_questions.items():
    print("=" * 80)
    print(question_id)
    print(question)

Q001
Which sales records describe WELIREG discussions related to renal cell carcinoma in territories where customer engagement or follow-up activity was also mentioned?
Q002
Find records where the sales discussion combines physician engagement, approved clinical information, and follow-up planning for a vaccine product.
Q003
Which records indicate regional or territory-level business opportunities while also referring to prescription trends and customer behavior?
Q004
Find records where customer discussions include market access or competitor comparison together with compliant promotional or scientific-literature activities.


## 5. Define the Embedding Models

The model definitions are centralized so the experiment logic does not need to change when comparing models.

`text-embedding-3-small` is the cost-efficient baseline.

`text-embedding-3-large` is the higher-capability OpenAI comparison.

We do not change the chunking strategy, questions or retrieval `k`.

In [5]:
embedding_models = {
    "EMB-001": {
        "name": "text-embedding-3-small",
        "provider": "OpenAI",
        "factory": lambda: OpenAIEmbeddings(
            model="text-embedding-3-small"
        ),
    },
    "EMB-002": {
        "name": "text-embedding-3-large",
        "provider": "OpenAI",
        "factory": lambda: OpenAIEmbeddings(
            model="text-embedding-3-large"
        ),
    },
}

for experiment_id, config in embedding_models.items():
    print(
        f"{experiment_id} | "
        f"{config['provider']} | "
        f"{config['name']}"
    )

EMB-001 | OpenAI | text-embedding-3-small
EMB-002 | OpenAI | text-embedding-3-large


## 6. Create the Embedding Experiment Function

For each model we:

1. Create a fresh Chroma vector store.
2. Embed the same 1,538 chunks.
3. Execute all four questions.
4. Retrieve the top 3 results.
5. Record rank, score and source row.

`k=3` is used here so Notebook 04 can separately investigate the effect of changing `k`.

In [6]:
def run_embedding_experiment(
    experiment_id,
    model_config,
    documents,
    questions,
    k=3,
):
    # Create a fresh embedding model for this experiment.
    embedding_model = model_config["factory"]()

    # Use a unique collection so the models do not share vectors.
    collection_name = f"embedding_eval_{experiment_id.lower()}"

    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        collection_name=collection_name,
    )

    rows = []

    for question_id, question in questions.items():
        results = vector_store.similarity_search_with_relevance_scores(
            question,
            k=k
        )

        for rank, (document, score) in enumerate(results, start=1):
            rows.append({
                "Experiment_ID": experiment_id,
                "Model": model_config["name"],
                "Provider": model_config["provider"],
                "Question_ID": question_id,
                "Rank": rank,
                "Score": float(score),
                "Source_Row": document.metadata.get("row"),
                "Content_Preview": document.page_content[:250],
            })

    return pd.DataFrame(rows)

## 7. Run the Embedding Experiments

Both models are evaluated against the same 300 documents, 1,538 chunks, four questions and `k=3`.

Only the embedding model changes.

In [7]:
all_embedding_results = []

for experiment_id, model_config in embedding_models.items():
    print("=" * 80)
    print(f"Running {experiment_id}: {model_config['name']}")

    result_df = run_embedding_experiment(
        experiment_id=experiment_id,
        model_config=model_config,
        documents=chunks,
        questions=evaluation_questions,
        k=3,
    )

    all_embedding_results.append(result_df)

    print(
        f"Completed {experiment_id} | "
        f"Results: {len(result_df)} rows"
    )

embedding_results_df = pd.concat(
    all_embedding_results,
    ignore_index=True
)

print("=" * 80)
print("All embedding experiments completed.")

Running EMB-001: text-embedding-3-small
Completed EMB-001 | Results: 12 rows
Running EMB-002: text-embedding-3-large
Completed EMB-002 | Results: 12 rows
All embedding experiments completed.


## 8. Inspect Retrieval Results

First inspect model, question, rank, score and source row.

The content preview remains available for deeper investigation.

In [8]:
display(
    embedding_results_df[
        [
            "Experiment_ID",
            "Model",
            "Question_ID",
            "Rank",
            "Score",
            "Source_Row",
        ]
    ]
)

,Experiment_ID,Model,Question_ID,Rank,Score,Source_Row
0,EMB-001,text-embedding-3-small,Q001,1,0.629833,102
1,EMB-001,text-embedding-3-small,Q001,2,0.629833,132
2,EMB-001,text-embedding-3-small,Q001,3,0.629823,13
3,EMB-001,text-embedding-3-small,Q002,1,0.457689,9
4,EMB-001,text-embedding-3-small,Q002,2,0.457689,154
5,EMB-001,text-embedding-3-small,Q002,3,0.457689,181
6,EMB-001,text-embedding-3-small,Q003,1,0.357033,9
7,EMB-001,text-embedding-3-small,Q003,2,0.357033,154
8,EMB-001,text-embedding-3-small,Q003,3,0.357033,271
9,EMB-001,text-embedding-3-small,Q004,1,0.281024,11


## 9. Compare Models Across Questions

For each model and question we calculate:

- Average score across the top 3 results
- Top-1 score

### Important

A relevance score is **not an accuracy percentage**.

For example, `0.82` does not mean 82% correct. It is a retrieval relevance signal.

In [9]:
score_summary = (
    embedding_results_df
    .groupby(
        ["Experiment_ID", "Model", "Question_ID"],
        as_index=False
    )
    .agg(
        Average_Top3_Score=("Score", "mean"),
        Top1_Score=("Score", "max"),
    )
)

display(score_summary)

,Experiment_ID,Model,Question_ID,Average_Top3_Score,Top1_Score
0,EMB-001,text-embedding-3-small,Q001,0.629829,0.629833
1,EMB-001,text-embedding-3-small,Q002,0.457689,0.457689
2,EMB-001,text-embedding-3-small,Q003,0.357033,0.357033
3,EMB-001,text-embedding-3-small,Q004,0.281024,0.281024
4,EMB-002,text-embedding-3-large,Q001,0.690597,0.690597
5,EMB-002,text-embedding-3-large,Q002,0.447196,0.447207
6,EMB-002,text-embedding-3-large,Q003,0.381081,0.381081
7,EMB-002,text-embedding-3-large,Q004,0.340038,0.340078


## 10. Model-Level Summary

The query-level results are aggregated to understand consistency across all four questions.

This is a screening metric, not a production benchmark.

In [10]:
model_summary = (
    score_summary
    .groupby(
        ["Experiment_ID", "Model"],
        as_index=False
    )
    .agg(
        Average_Query_Score=("Average_Top3_Score", "mean"),
        Average_Top1_Score=("Top1_Score", "mean"),
    )
    .sort_values(
        "Average_Query_Score",
        ascending=False
    )
)

display(model_summary)

,Experiment_ID,Model,Average_Query_Score,Average_Top1_Score
1,EMB-002,text-embedding-3-large,0.464728,0.464741
0,EMB-001,text-embedding-3-small,0.431394,0.431395


## 11. Percentage Difference vs Baseline

`EMB-001` (`text-embedding-3-small`) is the comparison baseline.

```text
(Model Score - Baseline Score)
-------------------------------- × 100
       Baseline Score
```

This is a relative retrieval-score difference, not an accuracy percentage.

In [11]:
baseline_id = "EMB-001"

baseline_row = model_summary[
    model_summary["Experiment_ID"] == baseline_id
]

if not baseline_row.empty:
    baseline_score = baseline_row.iloc[0]["Average_Query_Score"]

    model_summary["Difference_vs_EMB001_%"] = (
        (
            model_summary["Average_Query_Score"]
            - baseline_score
        )
        / baseline_score
        * 100
    )

display(model_summary)

,Experiment_ID,Model,Average_Query_Score,Average_Top1_Score,Difference_vs_EMB001_%
1,EMB-002,text-embedding-3-large,0.464728,0.464741,7.727089
0,EMB-001,text-embedding-3-small,0.431394,0.431395,0.000000


## 12. Question-Level Comparison

A model can perform well overall while struggling with one particular question.

We therefore compare both models question by question.

In [12]:
question_comparison = (
    score_summary
    .pivot(
        index="Question_ID",
        columns="Experiment_ID",
        values="Average_Top3_Score"
    )
)

display(question_comparison)

Experiment_ID,EMB-001,EMB-002
Question_ID,,
Q001,0.629829,0.690597
Q002,0.457689,0.447196
Q003,0.357033,0.381081
Q004,0.281024,0.340038


## 13. Find the Hardest Question for Each Model

The question with the lowest average top-three score is treated as the most difficult query for that model.

In [13]:
hardest_queries = (
    score_summary
    .sort_values(
        ["Experiment_ID", "Average_Top3_Score"]
    )
    .groupby("Experiment_ID")
    .first()
    .reset_index()
)

display(
    hardest_queries[
        [
            "Experiment_ID",
            "Model",
            "Question_ID",
            "Average_Top3_Score",
        ]
    ]
)

,Experiment_ID,Model,Question_ID,Average_Top3_Score
0,EMB-001,text-embedding-3-small,Q004,0.281024
1,EMB-002,text-embedding-3-large,Q004,0.340038


## 14. Inspect Retrieved Content for Q004

Scores alone are not enough.

Inspecting the actual retrieved records helps determine whether the retrieved text really addresses the business concepts in the question.

This is a simple manual relevance check and remains within the current course scope.

In [14]:
for experiment_id in embedding_models:

    inspection_results = embedding_results_df[
        (
            embedding_results_df["Experiment_ID"] == experiment_id
        )
        & (
            embedding_results_df["Question_ID"] == "Q004"
        )
    ].sort_values("Rank")

    print("=" * 80)
    print(f"Model: {embedding_models[experiment_id]['name']}")

    for _, row in inspection_results.iterrows():
        print("-" * 80)
        print(f"Rank  : {row['Rank']}")
        print(f"Score : {row['Score']:.4f}")
        print(f"Row   : {row['Source_Row']}")
        print("Preview:")
        print(row["Content_Preview"])

Model: text-embedding-3-small
--------------------------------------------------------------------------------
Rank  : 1
Score : 0.2810
Row   : 11
Preview:
patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature shar
--------------------------------------------------------------------------------
Rank  : 2
Score : 0.2810
Row   : 53
Preview:
patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature shar
--------------------------------------------------------------------------------
Rank  : 3
Score : 0.2810
Row   : 181
Preview:
patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician educa

## 15. Findings

### What We Evaluate

We look for:

- Consistency across all four questions
- Differences in retrieved source rows
- Differences in relevance scores
- Questions that are difficult for both models
- Cases where a numerical score does not necessarily mean useful business context

### Dataset Caveat

The dataset contains repeated or similar business-note patterns.

Therefore, multiple records can receive similar scores.

This should be presented as a controlled learning and comparison experiment, not a production benchmark.

## 16. Decision

Do not automatically select a model only because it has the highest numerical score.

Consider:

1. Model-level score consistency
2. Question-level behavior
3. Actual retrieved records
4. Cost and latency
5. Suitability for the intended solution

### Decision Template

After reviewing the generated outputs, record:

```text
Selected Model: text-embedding-3-large

Reason:
It achieved the highest overall average Top-3 and Top-1 retrieval
scores across the four evaluation questions, with an average Top-3
score of 0.464728 versus 0.431394 for text-embedding-3-small
(~7.73% higher).

Strongest Question:
Which sales records describe WELIREG discussions related to renal cell
carcinoma in territories where customer engagement or follow-up
activity was also mentioned?

Strongest Score:
0.690597

Weakest Question:
Find records where customer discussions include market access or
competitor comparison together with compliant promotional or
scientific-literature activities.

Weakest Score:
0.340038

Trade-off:
text-embedding-3-large provides better overall retrieval performance,
but it does not outperform text-embedding-3-small on every question
(Q002 performed slightly better with text-embedding-3-small).
It also has higher embedding cost than the smaller model.
```

The selected model becomes the initial embedding baseline for Notebook 04.

## 17. Persist Detailed Results

Detailed retrieval results are saved under:

```text
experiments/results/
```

Files:

- `embedding_experiment_results.csv`
- `embedding_model_summary.csv`

In [15]:
RESULTS_DIR = Path("../experiments/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_path = RESULTS_DIR / "embedding_experiment_results.csv"
summary_path = RESULTS_DIR / "embedding_model_summary.csv"

embedding_results_df.to_csv(
    results_path,
    index=False
)

model_summary.to_csv(
    summary_path,
    index=False
)

print(f"Detailed results saved to: {results_path}")
print(f"Model summary saved to: {summary_path}")

Detailed results saved to: ..\experiments\results\embedding_experiment_results.csv
Model summary saved to: ..\experiments\results\embedding_model_summary.csv


## 18. Update the Central Experiment Log

Embedding model summaries are also written to `experiments/experiment_log.csv`.

Existing EMB records are replaced when the notebook is rerun, preventing duplicate entries.

In [16]:
EXPERIMENT_LOG_PATH = "../experiments/experiment_log.csv"

embedding_log = model_summary.copy()

embedding_log["Area"] = "Embedding"
embedding_log["Configuration"] = embedding_log["Model"]
embedding_log["Question"] = "Q001-Q004"
embedding_log["Documents"] = len(data)
embedding_log["Chunks"] = len(chunks)
embedding_log["Top_K"] = 3
embedding_log["Results"] = (
    embedding_log["Average_Query_Score"]
    .round(4)
    .astype(str)
)
embedding_log["Observation"] = (
    "Compared retrieval behavior across four multi-criteria questions."
)
embedding_log["Conclusion"] = (
    "Review question-level results and retrieved content before "
    "selecting the embedding baseline."
)

log_columns = [
    "Experiment_ID",
    "Area",
    "Configuration",
    "Question",
    "Documents",
    "Chunks",
    "Top_K",
    "Results",
    "Observation",
    "Conclusion",
]

embedding_log = embedding_log[log_columns]

if (
    os.path.exists(EXPERIMENT_LOG_PATH)
    and os.path.getsize(EXPERIMENT_LOG_PATH) > 0
):
    existing_log = pd.read_csv(EXPERIMENT_LOG_PATH)

    if "Experiment_ID" in existing_log.columns:
        existing_log = existing_log[
            ~existing_log["Experiment_ID"].isin(
                list(embedding_models.keys())
            )
        ]

        combined_log = pd.concat(
            [existing_log, embedding_log],
            ignore_index=True
        )
    else:
        combined_log = embedding_log
else:
    combined_log = embedding_log

combined_log.to_csv(
    EXPERIMENT_LOG_PATH,
    index=False
)

print(f"Experiment log updated: {EXPERIMENT_LOG_PATH}")
print(f"Total logged experiments: {len(combined_log)}")

Experiment log updated: ../experiments/experiment_log.csv
Total logged experiments: 5


## 19. Final Validation

The notebook is complete when:

- 300 source documents are loaded
- 1,538 baseline chunks are generated
- Both embedding models execute
- All four questions are evaluated
- Each model/question combination returns three results
- Detailed results are persisted

In [17]:
assert len(data) == 300
assert len(chunks) == 1538
assert len(embedding_models) == 2
assert not embedding_results_df.empty

assert set(
    embedding_results_df["Question_ID"].unique()
) == set(evaluation_questions.keys())

results_per_question = (
    embedding_results_df
    .groupby(
        ["Experiment_ID", "Question_ID"]
    )
    .size()
)

assert (results_per_question == 3).all()

print("✓ Embedding experiment validation passed.")
print("✓ Models evaluated: 2")
print("✓ Questions evaluated: 4")
print("✓ Top-k used: 3")
print("✓ Detailed results persisted.")
print("✓ Ready for Notebook 04 — Similarity Search Experiments.")

✓ Embedding experiment validation passed.
✓ Models evaluated: 2
✓ Questions evaluated: 4
✓ Top-k used: 3
✓ Detailed results persisted.
✓ Ready for Notebook 04 — Similarity Search Experiments.


## 20. Handoff to Notebook 04

```text
300 Documents
      ↓
500 / 50 Chunking Baseline
      ↓
1,538 Chunks
      ↓
OpenAI Embedding Comparison
      ↓
text-embedding-3-small
text-embedding-3-large
      ↓
4 Multi-Criteria Questions
      ↓
Top-3 Similarity Search
      ↓
Score + Ranking + Source Comparison
      ↓
Notebook 04 — Similarity Search
```

Notebook 04 will focus specifically on retrieval behavior, including `k`, ranking, relevance scores and deeper inspection of retrieved results.